# GMM-token PFN: propagating a known d-dim displacement GMM

*Issue #23, branch `exp/23-gmm-prop`. An idea of the user's (2026-11-05), unrelated to the AIR line. Exploratory: one seed, one pilot.*

## Description

<!-- Written before the first run (2026-11-05); kept up to date. -->

**Idea.** A PFN maps a d-dim input to a binned (bar) distribution over a 1-dim output. A displacement $u(x)\in\mathbb R^d$ is a d-dim output, and bins over $\mathbb R^d$ need $B^d$ cells, which is infeasible beyond d ≈ 2. Instead:
- each **context token** sits at one of A's locations $x_i$ and carries a **GMM over the displacement** there, $q_i(u)=\sum_m w_{im}\,\mathcal N(u;\mu_{im},s_i^2 I)$ (as a probabilistic registration would give);
- PFN self-attention between context tokens and cross-attention from queries to context propagate this knowledge;
- the **output at any query** $x^*$ is a GMM over $u(x^*)$ (a mixture-density head with $K$ diagonal components: $K(2d+1)$ numbers, linear in d).

Later (not here), one samples displacements from the query's GMM, queries B's KV-cached PFN at $x^*+u$ and weights by the GMM to get a mixture over $y$. That needs a probabilistic registration, the output map h, and a weighting against the A-only predictive. **This experiment tests only the propagation, with the token GMMs given.**

**Question.** How close does a GMM-token PFN get to the exact Bayes predictive of the displacement at a new location, as a function of the dimension d and the number of context tokens n? And does the GMM head keep the *joint* multimodality across the d output dimensions that a per-dimension (bar-distribution) head cannot represent?

**Prior (one task).**
- Dimension d (one model per d ∈ {1, 2, 4, 8}); locations $x_i, x^*\sim\mathcal U[0,1]^d$.
- $M\in\{1,2,3\}$ displacement hypotheses with $p(M)=(0.4,0.3,0.3)$, weights $w\sim\text{Dir}(1_M)$.
- Each hypothesis is a field $u_m:[0,1]^d\to\mathbb R^d$, independent GP draws per output dimension, squared-exponential kernel, amplitude $\alpha=0.1$, length scale $\ell=\ell_0\sqrt d$ with $\ell_0\in\{0.2,0.4,0.8\}$ uniform (the $\sqrt d$ keeps typical pairwise correlations comparable across d).
- Token i: per-token noise $s_i\sim\log\mathcal U[0.005,0.05]$; components $(w_m,\ \mu_{im}=u_m(x_i)+s_i\varepsilon,\ s_i)$, **shuffled independently per token**, so a token's components carry no label; correspondence across tokens has to be inferred (from smoothness and from the weights, which are shared by the hypotheses).
- Target at a query: $m^*\sim\text{Cat}(w)$ independently per query, $u^*=u_{m^*}(x^*)$ (the noise-free displacement).
- n ~ U{0, …, 128} per training batch; 64 queries per task.

**Model.** Token = MLP(embedding of $x_i$ + DeepSets sum over its components of MLP$(w,\mu/\alpha,\log(s/\alpha))$); a learned null token is always in the context (n = 0 works). Pre-LN PFN layers: every token attends to the context tokens only (queries do not see each other). Head: $K=10$ diagonal Gaussians. Inputs and outputs are in units of α. 4 layers, width 128, 4 heads. Trained by the NLL of $u^*$; AdamW, lr 3e-4, cosine with warmup, 30k steps of 64 tasks.

**References on the same tasks (fixed evaluation set, CPU-generated).**
- **oracle**: the exact Bayes predictive given the *labelled* data (it knows which component of each token belongs to which hypothesis): mixture over ℓ (posterior from the GP evidence of all hypotheses) and over m (weights $w_m$) of the GP posteriors. Closed form, no MC error. With shuffled labels the true Bayes predictive also has to infer the labelling, so for M ≥ 2 this oracle is an **optimistic** bound (exact for M = 1).
- **oracle_fact**: the product over the d output dimensions of the oracle's 1-dim marginals: the best any per-dimension head (e.g. d bar distributions) could do, bin resolution aside. Equal to the oracle for d = 1 or M = 1 with one dominant ℓ.
- **prior**: $\mathcal N(0,\alpha^2 I)$, what n = 0 allows.

**What to look for.**
- Metric: NLL of $u^*$ at a query (joint over the d dimensions, nats; reported also per dimension), mean over 64 queries and 256 tasks per cell; cells n ∈ {0, 1, 2, 4, 8, 16, 32, 64, 128}.
- Gap Δ(d, n) = NLL_model − NLL_oracle with a 95 % bootstrap CI over tasks; Δ/d per dimension. Strata M = 1 (no correspondence problem) and M ≥ 2.
- Joint multimodality: NLL_model − NLL_oracle_fact on M ≥ 2 tasks for d ≥ 2.
- Sanity: at n = 0 the model must match the prior (Δ ≈ 0); the oracle's gain over the prior grows with n.
- Qualitative: d = 1 predictive density over x with the token GMMs; d = 2 predictive density over $u^*$ at a few queries, model against oracle.

**GO / NO-GO (exploratory, per d).**
- A cell (d, n) **propagates** if Δ/d ≤ 0.05 nats on all tasks and on the M ≥ 2 stratum.
- **GO** for the GMM-token propagation if every cell with d ≤ 4 propagates **and** for d ∈ {2, 4} the model beats oracle_fact on M ≥ 2 tasks at every n ≥ 8 (CI excludes 0).
- **INCONCLUSIVE** if this holds for d ≤ 2 only; **NO-GO** if d = 2 fails.
- d = 8 is descriptive: how the gap grows.

**Limits.**
- The token GMMs are given and exact in form (isotropic, shared weights across tokens); a real registration gives approximate, locally varying weights. Shared weights make correspondence easier than it will be.
- The oracle for M ≥ 2 is optimistic (labelled), so Δ there includes the unavoidable cost of not knowing the labelling.
- The prior is a GP on [0,1]^d; one seed; one model per d (no d-amortisation); n ≤ 128 (the training range).


In [ ]:
from meta_pfn.analysis import load_snapshot, paired_diff, show_header, summarize

SNAPSHOT = "results/gmm_prop/<sweep_id>"  # set after `just snapshot runs/gmm_prop/<sweep_id>`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

## Results

In [ ]:
# Aggregates use DONE members only; n_failed / n_started / n_missing say what is not in them yet.
summarize(snap.runs, "<metric>")

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.